In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import os
import zipfile
import base64

# Global Dictionairy

def get_size_premium(market_cap_billions):
    if pd.isna(market_cap_billions): return 0.02
    if market_cap_billions >= 50.0: return 0.00
    if market_cap_billions >= 10.0: return 0.005
    if market_cap_billions >= 2.0: return 0.012
    if market_cap_billions >= 0.3: return 0.025
    return 0.045

sector_adjustment_matrix = {
    'Technology': {'smooth_fcf': True, 'wacc_modifier': 0.00},
    'Utilities': {'smooth_fcf': False, 'wacc_modifier': -0.01}
}

sector_exit_multiples = {
    'Technology': 18.5, 'Consumer Cyclical': 12.0, 'Consumer Defensive': 14.0,
    'Communication Services': 15.0, 'Healthcare': 14.5, 'Financial Services': 10.0,
    'Energy': 7.5, 'Industrials': 12.5, 'Real Estate': 16.0, 'Utilities': 10.0, 'Basic Materials': 8.5
}

# Section 1: Grading Scales (Sector-Adaptive)

def grade_margin_adaptive(value, sector, metric_type='gross'):
    if pd.isna(value): return 0
    t1, t2, t3, t4 = 40, 30, 20, 10
    if metric_type != 'gross': t1, t2, t3, t4 = 20, 15, 10, 5
    if sector in ['Consumer Defensive', 'Consumer Cyclical']: t1, t2, t3, t4 = (t1/1.5), (t2/1.5), (t3/1.5), (t4/1.5)
    if value >= t1: return 10
    if value >= t2: return 8
    if value >= t3: return 6
    if value >= t4: return 4
    return 2

def grade_growth(value):
    if pd.isna(value): return 0
    if value >= 20: return 10
    elif value >= 10: return 8
    elif value >= 5: return 6
    elif value > 0: return 4
    return 0

def grade_valuation_pe(value):
    if pd.isna(value) or value <= 0: return 0
    if value <= 15: return 10
    elif value <= 20: return 8
    elif value <= 25: return 6
    elif value <= 35: return 4
    return 2

def grade_returns(value):
    if pd.isna(value): return 0
    if value >= 20: return 10
    elif value >= 15: return 8
    elif value >= 10: return 6
    elif value >= 5: return 4
    return 2

def get_final_rating(score):
    if score >= 8.5: return "Elite / Top-Tier"
    elif score >= 7.5: return "Strong Buy"
    elif score >= 6.5: return "Buy"
    elif score >= 5.5: return "Hold"
    return "Avoid"

# Section 2: Monte Carlo DCF

def generate_dcf_model(ticker, sector, current_price, current_fcf, current_ebitda, current_revenue, shares_out, net_debt, base_growth_rate, live_wacc, roic=None):
    if pd.isna(shares_out) or shares_out <= 0:
        return "N/A (Missing Shares Data)", 0, np.nan

    try:
        if live_wacc > 1.0: live_wacc = live_wacc / 100
        if shares_out < 10000 and pd.notna(current_revenue) and current_revenue > 1000000:
            shares_out = shares_out * 1000000

        if pd.notna(base_growth_rate) and base_growth_rate < 0: tgr = 0.00
        elif roic and pd.notna(roic) and roic > 20: tgr = 0.035
        else: tgr = 0.025

        effective_wacc = max(live_wacc, 0.08)
        base_g = min(max(base_growth_rate / 100, -0.05), 0.25) if pd.notna(base_growth_rate) else 0.05

        iterations = 1000
        rand_g = np.clip(np.random.normal(base_g, 0.015, iterations), 0.0, 0.25)
        rand_wacc = np.clip(np.random.normal(effective_wacc, 0.01, iterations), 0.08, 0.15)

        base_ebitda_mult = sector_exit_multiples.get(sector, 12.0)
        rand_ebitda_mult = np.random.normal(base_ebitda_mult, 1.5, iterations)

        implied_prices = []
        burn_rate_mode = pd.isna(current_fcf) or current_fcf <= 0 or pd.isna(current_ebitda) or current_ebitda <= 0

        for i in range(iterations):
            g_start = rand_g[i]
            wacc = rand_wacc[i]
            if wacc <= tgr: wacc = tgr + 0.001

            forecast_years = 10 if sector in ['Technology', 'Communication Services'] or (roic and roic > 20) else 5
            g_fade = [g_start - (g_start - tgr) * (yr / forecast_years) for yr in range(1, forecast_years + 1)]

            if burn_rate_mode:
                rev_proj = current_revenue
                for yr in range(forecast_years): rev_proj *= (1 + g_fade[yr])
                terminal_value = rev_proj * (rand_ebitda_mult[i] / 4)
                blended_ev = terminal_value / (1 + wacc)**forecast_years
            else:
                fcf_proj = current_fcf
                ebitda_proj = current_ebitda
                pv_fcf_sum = 0
                for yr in range(forecast_years):
                    fcf_proj *= (1 + g_fade[yr])
                    ebitda_proj *= (1 + g_fade[yr])
                    pv_fcf_sum += fcf_proj / (1 + wacc)**(yr+1)

                tv_gordon = (fcf_proj * (1 + tgr)) / (wacc - tgr)
                ev_gordon = pv_fcf_sum + (tv_gordon / (1 + wacc)**forecast_years)

                tv_multiple = ebitda_proj * rand_ebitda_mult[i]
                ev_multiple = pv_fcf_sum + (tv_multiple / (1 + wacc)**forecast_years)
                blended_ev = (ev_gordon + ev_multiple) / 2

            implied_equity = blended_ev - net_debt
            implied_price = max((implied_equity / shares_out), 0)
            implied_prices.append(implied_price)

        implied_prices = np.array(implied_prices)
        implied_prices = implied_prices[(implied_prices >= np.percentile(implied_prices, 2)) & (implied_prices <= np.percentile(implied_prices, 98))]

        mean_implied = np.median(implied_prices)
        prob_undervalued = np.sum(implied_prices > current_price) / len(implied_prices) * 100

        fig = plt.figure(figsize=(7, 4))
        plt.hist(implied_prices, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
        plt.axvline(current_price, color='red', linestyle='dashed', linewidth=2, label=f'Current: ${current_price:.2f}')
        plt.axvline(mean_implied, color='green', linestyle='dashed', linewidth=2, label=f'Mean: ${mean_implied:.2f}')
        plt.title(f"{ticker} Monte Carlo (1,000 Scenarios)", fontsize=12, fontweight='bold')
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{ticker}_DCF_Chart.png", bbox_inches='tight')
        plt.close(fig)

        return f"Mean: ${mean_implied:.2f} ({prob_undervalued:.0f}% Win)", prob_undervalued, mean_implied
    except Exception as e:
        return f"Error: {e}", 0, np.nan

# Section 3: Data Requirements

def run_master_pipeline(ticker_symbol, risk_free_rate):
    try:
        company = yf.Ticker(ticker_symbol)
        info = company.info
        sector = info.get('sector', 'Unknown')

        inc = company.financials.dropna(how='all').T.sort_index()
        bal = company.balance_sheet.dropna(how='all').T.sort_index()
        cf = company.cashflow.dropna(how='all').T.sort_index()

        latest_date = inc.index[-1]

        revenue = inc.loc[latest_date, 'Total Revenue']
        operating_income = inc.loc[latest_date].get('Operating Income', 0)
        ni_name = 'Net Income From Continuing Operation Net Minority Interest' if 'Net Income From Continuing Operation Net Minority Interest' in inc.columns else 'Net Income'
        net_income = inc.loc[latest_date, ni_name]

        total_assets = bal.loc[latest_date, 'Total Assets']
        current_liabilities = bal.loc[latest_date].get('Current Liabilities', np.nan)
        total_debt = bal.loc[latest_date].get('Total Debt', 0)
        cash_equiv = bal.loc[latest_date].get('Cash And Cash Equivalents', 0)
        net_debt = max(total_debt - cash_equiv, 0)

        market_cap = info.get('marketCap', np.nan)
        current_price = info.get('currentPrice', np.nan)
        shares_out = info.get('sharesOutstanding', np.nan)

        ebitda = info.get('ebitda', inc.loc[latest_date].get('EBITDA', np.nan))
        eps_growth = info.get('earningsGrowth', 0) * 100
        rev_growth_yoy = info.get('revenueGrowth', 0) * 100
        beta = info.get('beta', 1.0)

        equity_risk_premium = 0.055
        cost_of_equity = risk_free_rate + (beta * equity_risk_premium)
        cost_of_debt = 0.045
        tax_rate = inc.loc[latest_date].get('Tax Rate For Calcs', 0.21)
        safe_tax_rate = tax_rate if pd.notna(tax_rate) else 0.21

        total_capital = market_cap + total_debt if pd.notna(market_cap) else 1
        weight_e = market_cap / total_capital
        weight_d = total_debt / total_capital

        base_wacc = (weight_e * cost_of_equity) + (weight_d * cost_of_debt * (1 - safe_tax_rate))
        size_premium = get_size_premium(market_cap / 1e9) if pd.notna(market_cap) else 0.02
        live_wacc = max(base_wacc + size_premium, 0.07)
        dynamic_wacc_pct = live_wacc * 100

        fcf = cf.loc[latest_date].get('Free Cash Flow', np.nan) if not cf.empty else np.nan
        nopat = operating_income * (1 - safe_tax_rate)
        invested_cap = bal.loc[latest_date].get('Invested Capital', (total_assets - current_liabilities))
        roic = (nopat / invested_cap) * 100 if invested_cap > 0 else np.nan

        blended_growth_rate = (eps_growth * 0.7) + (rev_growth_yoy * 0.3) if pd.notna(eps_growth) else (rev_growth_yoy)

        dcf_status, mc_win_rate, mean_implied = generate_dcf_model(
            ticker_symbol, sector, current_price, fcf, ebitda, revenue,
            shares_out, net_debt, blended_growth_rate, live_wacc, roic
        )

        roic_wacc_spread = roic - dynamic_wacc_pct if pd.notna(roic) else np.nan
        passes_dual_hurdle = "Pass" if (pd.notna(roic) and roic > dynamic_wacc_pct) else "Fail"

        p1_score = grade_growth(rev_growth_yoy)
        p2_score = grade_returns(roic)
        p3_score = grade_valuation_pe(info.get('trailingPE', np.nan))
        safe_score = (p1_score + p2_score + p3_score) / 3 * 10

        base_conviction = (safe_score * 0.4) + (mc_win_rate * 0.6)
        if passes_dual_hurdle == "Fail": base_conviction -= 15
        conviction_score = round(max(0, min(base_conviction, 100)), 1)

        master_grade = "A (Buy)" if conviction_score >= 70 else "B (Hold)" if conviction_score >= 50 else "C (Avoid)"

        return {
            'Ticker': ticker_symbol,
            'Sector': sector,
            'Master Grade': master_grade,
            'Conviction Score (0-100)': conviction_score,
            'Final Rating': get_final_rating(safe_score / 10),
            'MC DCF Model': dcf_status,
            'Live WACC (%)': round(dynamic_wacc_pct, 2),
            'Beta': round(beta, 2),
            'ROIC (%)': round(roic, 2) if pd.notna(roic) else "N/A",
            'Dual Hurdle': passes_dual_hurdle,
            'ROIC-WACC Spread (%)': round(roic_wacc_spread, 2) if pd.notna(roic_wacc_spread) else "N/A",
            'Rev Growth (%)': round(rev_growth_yoy, 2),
            'P/E Ratio': round(info.get('trailingPE', np.nan), 2)
        }
    except Exception as e:
        return {"Ticker": ticker_symbol, "Error": str(e)}

# Section 4: Execution

target_stocks = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'HD', 'PG', 'CAT', 'XOM', 'NEE', 'DIS']

print("Pinging Yahoo Finance 10-Yr Treasury Yield (Risk-Free Rate)")
try:
    tnx = yf.Ticker("^TNX")
    current_risk_free_rate = tnx.history(period="1d")['Close'].iloc[-1] / 100
    print(f" U.S. Risk-Free Rate Locked: {current_risk_free_rate * 100:.2f}%")
except:
    current_risk_free_rate = 0.045
    print(" Using fallback Risk-Free Rate of 4.50%")

results = []
for ticker in target_stocks:
    print(f"DCF Generated {ticker}")
    data = run_master_pipeline(ticker, current_risk_free_rate)
    if data and "Error" not in data:
        results.append(data)
    time.sleep(1.0)

print("\n Analysis Complete \n")

if results:
    safe_database = pd.DataFrame(results)
    safe_database = safe_database.sort_values(by='Conviction Score (0-100)', ascending=False).reset_index(drop=True)

    csv_filename = "SAFE.csv"
    safe_database.to_csv(csv_filename, index=False)

    zip_filename = "SAFE_DCF_Charts.zip"
    chart_files = [f for f in os.listdir('.') if f.endswith("_DCF_Chart.png")]
    zip_successful = False
    if chart_files:
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for file in chart_files: zipf.write(file)
        zip_successful = True

    def create_download_button(filename, button_text, color):
        if not os.path.exists(filename): return ""
        with open(filename, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        payload = f"data:application/octet-stream;base64,{b64}"
        return f'<a download="{filename}" href="{payload}" target="_blank" style="display: inline-block; padding: 12px 24px; margin: 10px 10px 10px 0; background-color: {color}; color: white; text-decoration: none; border-radius: 6px; font-weight: bold; font-family: Arial, sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">{button_text}</a>'

    print(" Finished File Generation")
    html_buttons = create_download_button(csv_filename, "Download Database (CSV)", "#2ecc71")
    if zip_successful:
        html_buttons += create_download_button(zip_filename, "Download Monte Carlo Charts (ZIP)", "#3498db")
    display(HTML(html_buttons))

    pd.set_option('display.max_columns', None)
    display(safe_database)